In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("ABS Tech Case 2026_Data.xlsx")

# Normalize variables (0–1 scale)
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

# --- Technical abilities (50%)
technical = (
    0.15 * normalize(df["TechLev"]) +
    0.10 * normalize(df["TrainHours"]) +
    0.05 * normalize((df["AIUse"] + df["AIConf"]) / 2) +
    0.10 * normalize(df["InnoCont"]) +
    0.10 * normalize(df["SpecialProjectsCount"])
)

# --- Personal abilities (25%)
personal = (
    0.05 * normalize(df["EngagementSurvey"]) +
    0.05 * normalize(df["WLF"]) +
    0.15 * normalize(df["PerfScore"]) +
    0.05 * (1 - normalize(df["JobStr"]))  # lower stress is better
)

# --- Interpersonal abilities (25%)
interpersonal = (
    0.12 * normalize((df["Feedback"] + df["Trust"]) / 2) +
    0.06 * normalize(df["TeamIden"]) +
    0.05 * normalize(df["Network"]) +
    0.02 * normalize(df["ProjColl"])
)

# Final Talent Score
df["TalentScore"] = technical + personal + interpersonal

# Identify Talent (Top 20%)
threshold = df["TalentScore"].quantile(0.80)
df["IdentifiedTalent"] = (df["TalentScore"] >= threshold).astype(int)

# Compare Turnover Rates
overall_turnover = df["Termd"].mean()
talent_turnover = df[df["IdentifiedTalent"] == 1]["Termd"].mean()
non_talent_turnover = df[df["IdentifiedTalent"] == 0]["Termd"].mean()

print("Overall Turnover:", overall_turnover)
print("Talent Turnover:", talent_turnover)
print("Non-Talent Turnover:", non_talent_turnover)

# Turnover Trend Over Time
df["DateofTermination"] = pd.to_datetime(df["DateofTermination"], errors="coerce")
df["TerminationYear"] = df["DateofTermination"].dt.year

turnover_by_year = df[df["Termd"] == 1].groupby("TerminationYear").size()

# Plot trend
plt.figure()
turnover_by_year.plot(kind="line")
plt.title("Turnover Trend Over Time")
plt.xlabel("Year")
plt.ylabel("Number of Terminations")
plt.show()